# ImageNet-Grid Vis Heads: Cleaner Ground Truth, and Are They General?

A third Vis-Head dataset, built to fix a weakness in the other two:

- **Comics** — clean regions (panels), but hand-drawn, fixed content per strip, and the
  query is purely *positional/ordinal* ("what's in the 3rd panel?"), never names an
  object.
- **COCO** — real, cluttered natural photos with an unambiguous target object, but the
  target region can be tiny and the background is uncontrolled.
- **ImageNet-grid (this notebook)** — tiles `rows x cols` distinct, single-object
  ImageNet validation images into one grid. Each cell is a clean photograph of exactly
  one object with no clutter (like COCO's photographic realism), but the cell
  boundaries are as controlled as comic panels (like comics' clean regions), and — the
  key new property — **the object in each cell is re-randomized on every sample**, so
  we can ask a question comics and COCO can't cleanly answer:

  > Are vis heads specific to particular *objects*, specific to particular *prompt
  > wording*, or genuinely general — the same heads regardless of what's being asked
  > for or what it looks like?

This notebook:

1. **Builds the ImageNet-grid dataset** (`vis_head/imagenet_grid.py`) and visualizes
   a sample.
2. **Discovery** — vir scores on ImageNet-grid, compared against the already-computed
   comics/COCO normalized scores from `compare_comics_vs_coco_vis_heads.ipynb`
   (loaded from `logs/vis_head_discovery_compare_datasets_{comics,coco}/`, no rerun
   needed) — a three-way comparison of raw + area-normalized vir scores.
3. **Causal-effect sweep** — a shared top-head pool ablated across all three datasets,
   to see which dataset's heads are the most *causally* load-bearing, not just
   correlated with the target (semantic-similarity effect size, as in the other
   comparison notebooks).
4. **Object generalization** — split ImageNet-grid samples into two random halves
   (near-disjoint object categories by construction) and check whether the top heads
   found on one half predict the top heads on the other.
5. **Prompt/verb generalization** — fix the target, vary only the instruction's
   wording (`Find/Locate/Where is/Point to/Identify the {name}.`, plus a position-only
   phrasing with no object name) and check whether the same heads show up regardless
   of phrasing.
6. **Verdict.**

**Prerequisites**
- ImageNet under `VIR_IMAGENET_ROOT` (`val/<wnid>/*.JPEG` + `meta.bin`).
- Comics and a built COCO vir dataset, plus the cached results from
  `compare_comics_vs_coco_vis_heads.ipynb` (`logs/vis_head_discovery_compare_datasets_*`).
- A GPU with enough memory for one Qwen-VL checkpoint.
- Run from the repository root so `vis_head` imports resolve.


In [1]:
%matplotlib inline
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vis_head").exists():
    raise RuntimeError("Run this notebook from the repository root (vis-head/).")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from scipy import stats
from tqdm.auto import tqdm

from vis_head.common import (
    DEFAULT_COMICS_ROOT,
    DEFAULT_MODEL_ID,
    DEFAULT_N_PANELS,
    DEFAULT_SEED,
    dump_json,
    make_output_paths,
    seed_everything,
)
from vis_head.coco import DEFAULT_COCO_OUTPUT_DIR, load_coco_vis_head_dataset
from vis_head.data import build_strip, list_comic_dirs
from vis_head.vir import (
    aggregate_region_attention,
    collect_last_query_attentions,
    panel_query_prompt,
    panel_token_fractions,
    rank_heads_by_score,
    save_head_ranking,
)
from vis_head.imagenet_grid import (
    DEFAULT_IMAGENET_ROOT,
    PROMPT_TEMPLATES,
    draw_cell_labels,
    list_val_class_dirs,
    load_class_names,
    ordinal_prompt,
    sample_grid,
)
from vis_head.judge import bootstrap_ci, semantic_similarity
from vis_head.modeling import (
    decode_generated_text,
    find_image_token_range,
    load_model_and_processor,
    model_dims,
    prepare_inputs,
    run_generation,
)
from vis_head.plots import add_overlay_colorbar, draw_attention_overlay, save_figure_pdf
from vis_head.regions import (
    assign_grid_cells_to_tokens,
    assign_panels_to_tokens,
    bbox_to_token_positions,
    get_merged_grid_shape,
    region_positions_from_ids,
)
from vis_head.steering import group_heads_by_layer, make_static_attention_mask_hook, register_mask_hooks, remove_handles

EPS = 1e-8


In [2]:
# ----------------------------- configuration -----------------------------
MODEL_ID = DEFAULT_MODEL_ID     # must match the checkpoint used for the cached
                                 # comics/coco results below, so the shared head
                                 # pool in Part 3 is comparing the same heads.
DEVICE = "cuda:0"
SEED = DEFAULT_SEED

IMAGENET_ROOT = DEFAULT_IMAGENET_ROOT
ROWS, COLS = 2, 2                # 4 distinct objects per grid
CELL_SIZE = 256
N_CELLS = ROWS * COLS

N_DISCOVERY_SAMPLES = 150        # imagenet grids for Part 2/3 discovery + Part 4 object-generalization
N_CAUSAL_SAMPLES = 30            # per dataset, for the causal-effect sweep
HEAD_BUDGETS = [30, 10]
N_VERB_GRIDS = 30                # grids reused across all prompt phrasings in Part 5

COMICS_ROOT = Path(DEFAULT_COMICS_ROOT)
N_PANELS = DEFAULT_N_PANELS
COCO_DATASET_DIR = DEFAULT_COCO_OUTPUT_DIR
CACHED_COMPARE_LOGS = REPO_ROOT / "logs" / "vis_head_discovery_compare_datasets_{tag}"

comic_dirs = list_comic_dirs(COMICS_ROOT, n_panels=N_PANELS)[:N_CAUSAL_SAMPLES]
coco_dataset = load_coco_vis_head_dataset(COCO_DATASET_DIR)
coco_indices = list(range(min(N_CAUSAL_SAMPLES, len(coco_dataset))))

imagenet_class_dirs = list_val_class_dirs(IMAGENET_ROOT)
imagenet_class_names = load_class_names(IMAGENET_ROOT)
print(f"Model         : {MODEL_ID}")
print(f"ImageNet root : {IMAGENET_ROOT}  ({len(imagenet_class_dirs)} classes)")
print(f"Grid          : {ROWS}x{COLS} = {N_CELLS} cells, {CELL_SIZE}px/cell")
print(f"Comics        : {len(comic_dirs)} strips (causal-effect sample)")
print(f"COCO          : {len(coco_indices)} samples (causal-effect sample)")


Model         : Qwen/Qwen3-VL-8B-Instruct
ImageNet root : /mnt/abka03/raw_data_download/imagenet  (1000 classes)
Grid          : 2x2 = 4 cells, 256px/cell
Comics        : 30 strips (causal-effect sample)
COCO          : 30 samples (causal-effect sample)


## Part 1 — Build and inspect an ImageNet-grid sample

Four unrelated, distinct-category photographs tiled into one image, with a
"Find the {name}." instruction pointing at one cell — the model gets the full grid
image and the instruction only; the cell boundaries/labels below are for our own
sanity-checking, never shown to the model.

In [3]:
rng = np.random.RandomState(SEED)
demo_grid = sample_grid(
    imagenet_root=IMAGENET_ROOT, rows=ROWS, cols=COLS, cell_size=CELL_SIZE,
    rng=rng, class_dirs=imagenet_class_dirs, class_names=imagenet_class_names,
)
target_cell = int(rng.randint(N_CELLS))
prompt = PROMPT_TEMPLATES["find"].format(name=demo_grid.cell_names[target_cell])

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(demo_grid.grid)
axes[0].set_title(f'Model input\n(instruction: "{prompt}")')
axes[0].axis("off")
axes[1].imshow(draw_cell_labels(demo_grid))
axes[1].set_title(f"Cell contents (hidden ground truth): {demo_grid.cell_names}")
axes[1].axis("off")
plt.show()
print(f"Target cell (1-based): {target_cell + 1}  ->  {demo_grid.cell_names[target_cell]}")


Target cell (1-based): 2  ->  pop bottle


## Part 2 — Discovery on ImageNet-grid

One `"Find the {name}."` query per grid, target cell randomized each sample (so the
discovery pass itself already varies both the target object *and* its position).
Stores each sample's raw score matrix too (not just the running sum) — Part 4 reuses
these per-sample scores for the object-generalization split-half test with no extra
model calls.

In [4]:
model, processor = load_model_and_processor(model_id=MODEL_ID, device=DEVICE)
n_layers, n_heads, spatial_merge = model_dims(model)
print(f"{n_layers} layers x {n_heads} heads")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

36 layers x 32 heads


In [5]:
def discover_vis_head_imagenet(model, processor, n_samples: int, rows: int, cols: int, cell_size: int,
                                  imagenet_class_dirs, imagenet_class_names, seed: int,
                                  template: str = "find", fixed_grids=None):
    """One `"Find the {name}."` query per grid; returns (raw_sum, normalized_sum,
    per_sample_scores, valid_samples). `fixed_grids`, if given, is a list of
    (grid, target_cell) reused instead of sampling fresh ones (for the
    verb-generalization test in Part 5, which needs the same grids across templates).
    """
    rng = np.random.RandomState(seed)
    n_cells = rows * cols
    raw_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    normalized_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    per_sample_scores = []
    valid_samples = 0

    iterator = fixed_grids if fixed_grids is not None else range(n_samples)
    for item in tqdm(iterator, total=n_samples, desc=f"Vir discovery [imagenet/{template}]", leave=False):
        if fixed_grids is not None:
            grid, target_cell = item
        else:
            grid = sample_grid(rows=rows, cols=cols, cell_size=cell_size, rng=rng,
                                class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
            target_cell = int(rng.randint(n_cells))
        prompt = PROMPT_TEMPLATES[template].format(name=grid.cell_names[target_cell])
        try:
            inputs = prepare_inputs(processor, grid.grid, prompt, DEVICE)
            region_ids, _ = assign_grid_cells_to_tokens(
                image_grid_thw=inputs["image_grid_thw"], rows=rows, cols=cols, spatial_merge=spatial_merge,
            )
            attn_at_query = collect_last_query_attentions(model, inputs)
            region_attention = aggregate_region_attention(
                attn_at_query=attn_at_query, inputs=inputs, processor=processor,
                region_ids=region_ids, n_regions=n_cells,
            )
            raw = region_attention[:, :, target_cell]
            fraction = panel_token_fractions(region_ids, n_cells)[target_cell]
        except Exception as exc:
            print(f"Skipping grid {grid.name}: {exc}")
            continue
        raw_sum += raw
        normalized_sum += raw / max(fraction, EPS)
        per_sample_scores.append(raw)
        valid_samples += 1

    if valid_samples == 0:
        raise RuntimeError("No valid ImageNet-grid samples processed.")
    return (raw_sum / valid_samples).astype(np.float32), (normalized_sum / valid_samples).astype(np.float32), \
           per_sample_scores, valid_samples


seed_everything(SEED)
imagenet_scores, imagenet_scores_norm, imagenet_per_sample, n_imagenet_valid = discover_vis_head_imagenet(
    model, processor, N_DISCOVERY_SAMPLES, ROWS, COLS, CELL_SIZE, imagenet_class_dirs, imagenet_class_names, SEED,
)
print(f"valid samples: {n_imagenet_valid}/{N_DISCOVERY_SAMPLES}")


Vir discovery [imagenet/find]:   0%|          | 0/150 [00:00<?, ?it/s]

valid samples: 150/150


In [6]:
imagenet_ranked = rank_heads_by_score(imagenet_scores)
imagenet_ranked_norm = rank_heads_by_score(imagenet_scores_norm)

outputs = make_output_paths("vis_head_discovery_compare_datasets_imagenet")
np.save(outputs.logs_dir / "vis_head_scores.npy", imagenet_scores)
np.save(outputs.logs_dir / "vis_head_scores_normalized.npy", imagenet_scores_norm)
save_head_ranking(outputs.logs_dir / "vis_head_ranking.json", imagenet_ranked)
save_head_ranking(outputs.logs_dir / "vis_head_ranking_normalized.json", imagenet_ranked_norm)
dump_json(outputs.logs_dir / "summary.json", {
    "model_id": MODEL_ID, "dataset_source": "imagenet_grid", "rows": ROWS, "cols": COLS,
    "n_valid_samples": n_imagenet_valid, "n_layers": n_layers, "n_heads": n_heads,
    "top_heads": imagenet_ranked[:20], "top_heads_normalized": imagenet_ranked_norm[:20],
})
print(f"Saved to {outputs.logs_dir}")


Saved to /mnt/abka03/Projects/vis-head/logs/vis_head_discovery_compare_datasets_imagenet


## Part 3 — Three-way comparison: comics vs. COCO vs. ImageNet-grid

Loads the already-computed comics/COCO normalized scores from
`compare_comics_vs_coco_vis_heads.ipynb`'s output (same checkpoint, no rerun) and
compares all three side by side — this is the "which dataset gives cleaner vir
heads" question, at the attention-score level.

In [7]:
def load_cached_scores(tag: str) -> tuple[np.ndarray, np.ndarray]:
    log_dir = Path(str(CACHED_COMPARE_LOGS).format(tag=tag))
    raw = np.load(log_dir / "vis_head_scores.npy")
    norm = np.load(log_dir / "vis_head_scores_normalized.npy")
    return raw, norm

comic_scores, comic_scores_norm = load_cached_scores("comics")
coco_scores, coco_scores_norm = load_cached_scores("coco")

assert comic_scores.shape == coco_scores.shape == imagenet_scores.shape, (
    "Cached comics/coco scores were computed on a different model shape than "
    f"{MODEL_ID} — rerun compare_comics_vs_coco_vis_heads.ipynb with the same MODEL_ID first."
)

datasets = {"comics": (comic_scores, comic_scores_norm), "coco": (coco_scores, coco_scores_norm),
            "imagenet": (imagenet_scores, imagenet_scores_norm)}

print(f"{'dataset':>9s}  {'raw mean':>9s}  {'raw max':>9s}  {'norm mean':>10s}  {'norm max':>9s}")
for name, (raw, norm) in datasets.items():
    print(f"{name:>9s}  {raw.mean():9.5f}  {raw.max():9.5f}  {norm.mean():10.3f}  {norm.max():9.3f}")


  dataset   raw mean    raw max   norm mean   norm max
   comics    0.04759    0.71596       0.286      4.296
     coco    0.00723    0.13647       0.083      2.872
 imagenet    0.02527    0.28083       0.101      1.123


In [8]:
def top_k_head_set(ranked: list[dict], k: int) -> set:
    return {(row["layer"], row["head"]) for row in ranked[:k]}

pairs = [("comics", "coco"), ("comics", "imagenet"), ("coco", "imagenet")]
rankings_norm = {"comics": rank_heads_by_score(comic_scores_norm), "coco": rank_heads_by_score(coco_scores_norm),
                  "imagenet": imagenet_ranked_norm}

print("Pairwise top-100 overlap and Spearman rho (NORMALIZED scores):")
for a, b in pairs:
    overlap = len(top_k_head_set(rankings_norm[a], 100) & top_k_head_set(rankings_norm[b], 100))
    rho = stats.spearmanr(datasets[a][1].reshape(-1), datasets[b][1].reshape(-1)).correlation
    print(f"  {a:>9s} vs {b:<9s}  top-100 overlap: {overlap:3d}/100  ({overlap}%)   Spearman rho: {rho:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharex=True, sharey=True)
vmax = np.percentile(np.concatenate([d[1].reshape(-1) for d in datasets.values()]), 99)
for ax, (name, (raw, norm)) in zip(axes, datasets.items()):
    im = ax.imshow(norm, aspect="auto", cmap="magma", vmin=0, vmax=vmax)
    ax.set_title(f"{name} — normalized vir score")
    ax.set_xlabel("Head")
axes[0].set_ylabel("Layer")
fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02, label="Normalized vir score (x chance)")
save_figure_pdf(fig, outputs.figures_dir / "three_way_vis_head_score_maps.pdf")
plt.show()


Pairwise top-100 overlap and Spearman rho (NORMALIZED scores):
     comics vs coco       top-100 overlap:  50/100  (50%)   Spearman rho: 0.768
     comics vs imagenet   top-100 overlap:  40/100  (40%)   Spearman rho: 0.792
       coco vs imagenet   top-100 overlap:  67/100  (67%)   Spearman rho: 0.933


## Part 4 — Causal-effect sweep (three-way)

Shared top-head pool, ranked by the average of all three datasets' normalized scores
(so the same heads are tested everywhere), ablated at each head budget on comics,
COCO, and ImageNet-grid — the semantic-similarity effect size
(`vis_head/judge.py`, as in the other comparison notebooks) says which dataset's
task actually *depends* on those heads, as opposed to merely correlating with them.

In [9]:
combined_norm_score = (comic_scores_norm.astype(np.float64) + coco_scores_norm.astype(np.float64)
                        + imagenet_scores_norm.astype(np.float64)) / 3.0
combined_ranked = rank_heads_by_score(combined_norm_score)
print(f"Shared head pool (3-way combined normalized score), sweeping budgets {HEAD_BUDGETS}")


def ablation_effect(model, processor, image, prompt: str, heads_by_layer: dict,
                     target_positions: list[int], max_new_tokens: int = 40) -> dict:
    inputs = prepare_inputs(processor, image, prompt, DEVICE)
    prompt_length = int(inputs["input_ids"].shape[1])
    baseline_sequences = run_generation(model=model, inputs=inputs, max_new_tokens=max_new_tokens)
    baseline_text = decode_generated_text(processor, baseline_sequences, prompt_length)

    hook_by_layer = {
        layer_idx: make_static_attention_mask_hook(
            head_indices=heads, suppress_positions=target_positions, boost_positions=[],
            n_query_heads=n_heads, device=DEVICE, decode_only=False, pad_with_suppress=False,
        )
        for layer_idx, heads in heads_by_layer.items()
    }
    ablate_inputs = prepare_inputs(processor, image, prompt, DEVICE)
    handles = register_mask_hooks(model, hook_by_layer)
    try:
        ablated_sequences = run_generation(model=model, inputs=ablate_inputs, max_new_tokens=max_new_tokens)
    finally:
        remove_handles(handles)
    ablated_text = decode_generated_text(processor, ablated_sequences, prompt_length)

    similarity = semantic_similarity(ablated_text, baseline_text, device="cpu")
    return {"changed": similarity < 0.85, "effect": 1.0 - similarity}


def causal_comics(heads_by_layer, n_samples):
    rng = np.random.RandomState(SEED)
    results = []
    for comic_dir in tqdm(comic_dirs[:n_samples], desc="Causal [comics]", leave=False):
        strip = build_strip(comic_dir, n_panels=N_PANELS)
        target_panel = int(rng.randint(N_PANELS))
        prompt = panel_query_prompt(target_panel + 1, n_panels=N_PANELS)
        try:
            inputs = prepare_inputs(processor, strip.strip, prompt, DEVICE)
            img_start, _ = find_image_token_range(inputs, processor)
            region_ids, _, _ = assign_panels_to_tokens(
                image_grid_thw=inputs["image_grid_thw"], panel_widths=strip.panel_widths, spatial_merge=spatial_merge)
            panel_positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_PANELS)
            results.append(ablation_effect(model, processor, strip.strip, prompt, heads_by_layer, panel_positions[target_panel]))
        except Exception as exc:
            print(f"Skipping {strip.name}: {exc}")
    return results


def causal_coco(heads_by_layer, n_samples):
    results = []
    for idx in tqdm(coco_indices[:n_samples], desc="Causal [coco]", leave=False):
        meta, gt = coco_dataset[idx]
        try:
            image = Image.open(meta["image_path"]).convert("RGB")
            inputs = prepare_inputs(processor, image, meta["instruction"], DEVICE)
            img_start, _ = find_image_token_range(inputs, processor)
            grid_shape = get_merged_grid_shape(inputs["image_grid_thw"], spatial_merge)
            x, y, w, h = gt["bbox"]
            _, target_positions = bbox_to_token_positions((x, y, x + w, y + h), grid_shape, image.size, img_start)
            results.append(ablation_effect(model, processor, image, meta["instruction"], heads_by_layer, target_positions))
        except Exception as exc:
            print(f"Skipping sample {idx}: {exc}")
    return results


def causal_imagenet(heads_by_layer, n_samples):
    rng = np.random.RandomState(SEED + 1)
    results = []
    for _ in tqdm(range(n_samples), desc="Causal [imagenet]", leave=False):
        grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng,
                            class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
        target_cell = int(rng.randint(N_CELLS))
        prompt = PROMPT_TEMPLATES["find"].format(name=grid.cell_names[target_cell])
        try:
            inputs = prepare_inputs(processor, grid.grid, prompt, DEVICE)
            img_start, _ = find_image_token_range(inputs, processor)
            region_ids, _ = assign_grid_cells_to_tokens(
                image_grid_thw=inputs["image_grid_thw"], rows=ROWS, cols=COLS, spatial_merge=spatial_merge)
            positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
            results.append(ablation_effect(model, processor, grid.grid, prompt, heads_by_layer, positions[target_cell]))
        except Exception as exc:
            print(f"Skipping grid: {exc}")
    return results


causal_results = {}
for budget in HEAD_BUDGETS:
    heads_by_layer = group_heads_by_layer([(row["layer"], row["head"]) for row in combined_ranked[:budget]])
    print(f"\n=== head budget = {budget} ===")
    causal_results[budget] = {
        "comics": causal_comics(heads_by_layer, N_CAUSAL_SAMPLES),
        "coco": causal_coco(heads_by_layer, N_CAUSAL_SAMPLES),
        "imagenet": causal_imagenet(heads_by_layer, N_CAUSAL_SAMPLES),
    }


Shared head pool (3-way combined normalized score), sweeping budgets [30, 10]

=== head budget = 30 ===


Causal [comics]:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/776 [00:00<?, ?it/s]

Causal [coco]:   0%|          | 0/30 [00:00<?, ?it/s]

Causal [imagenet]:   0%|          | 0/30 [00:00<?, ?it/s]


=== head budget = 10 ===


Causal [comics]:   0%|          | 0/30 [00:00<?, ?it/s]

Causal [coco]:   0%|          | 0/30 [00:00<?, ?it/s]

Causal [imagenet]:   0%|          | 0/30 [00:00<?, ?it/s]

In [10]:
print(f"{'heads':>6s}  {'dataset':>9s}  {'n':>4s}  {'change rate':>12s}  {'95% CI':>16s}  {'mean effect':>12s}")
for budget in HEAD_BUDGETS:
    for name in ("comics", "coco", "imagenet"):
        res = causal_results[budget][name]
        changed = [r["changed"] for r in res]
        effects = [r["effect"] for r in res]
        ci = bootstrap_ci(changed)
        print(f"{budget:6d}  {name:>9s}  {ci['n']:4d}  {ci['accuracy']:12.3f}  "
              f"[{ci['ci_low']:.3f}, {ci['ci_high']:.3f}]  {np.mean(effects):12.3f}")
    all_effects = {name: [r["effect"] for r in causal_results[budget][name]] for name in ("comics", "coco", "imagenet")}
    for a, b in [("imagenet", "comics"), ("imagenet", "coco")]:
        mw = stats.mannwhitneyu(all_effects[a], all_effects[b], alternative="two-sided")
        higher = a if np.mean(all_effects[a]) > np.mean(all_effects[b]) else b
        print(f"         Mann-Whitney ({a} vs {b}): U={mw.statistic:.1f}  p={mw.pvalue:.3e}  -> larger effect on {higher}")

fig, ax = plt.subplots(figsize=(6, 5))
for name, color in (("comics", "tab:orange"), ("coco", "tab:green"), ("imagenet", "tab:purple")):
    means = [np.mean([r["effect"] for r in causal_results[b][name]]) for b in HEAD_BUDGETS]
    ax.plot(HEAD_BUDGETS, means, marker="o", label=name, color=color)
ax.set_xlabel("Ablated heads"); ax.set_ylabel("Mean causal effect (1 - semantic similarity)")
ax.set_title("Causal effect vs. head budget, by dataset"); ax.set_ylim(0, 1.05); ax.legend()
save_figure_pdf(fig, outputs.figures_dir / "three_way_causal_effect.pdf")
plt.show()


 heads    dataset     n   change rate            95% CI   mean effect
    30     comics    30         0.633  [0.467, 0.800]         0.345
    30       coco    30         0.867  [0.733, 0.967]         0.483
    30   imagenet    30         0.767  [0.600, 0.900]         0.404
         Mann-Whitney (imagenet vs comics): U=539.0  p=1.907e-01  -> larger effect on imagenet
         Mann-Whitney (imagenet vs coco): U=379.0  p=2.973e-01  -> larger effect on coco
    10     comics    30         0.667  [0.500, 0.833]         0.360
    10       coco    30         0.800  [0.633, 0.933]         0.501
    10   imagenet    30         0.800  [0.667, 0.933]         0.438
         Mann-Whitney (imagenet vs comics): U=570.0  p=7.727e-02  -> larger effect on imagenet
         Mann-Whitney (imagenet vs coco): U=359.0  p=1.809e-01  -> larger effect on coco


## Part 5 — Object generalization

`imagenet_per_sample` (from Part 2) holds one raw score matrix per discovery sample,
each with an *independently, randomly chosen* target object (near-certainly disjoint
category sets between any two random halves of 150 samples drawn from 1000 classes).
Split into two halves and check whether the top heads found on one half's objects
predict the top heads on the other half's — if they do, vis heads aren't tied to
specific object identities.

In [11]:
per_sample = np.stack(imagenet_per_sample)   # (n_samples, n_layers, n_heads)
rng = np.random.RandomState(SEED)
perm = rng.permutation(len(per_sample))
half_a, half_b = perm[: len(perm) // 2], perm[len(perm) // 2 :]

scores_a = per_sample[half_a].mean(axis=0)
scores_b = per_sample[half_b].mean(axis=0)
ranked_a, ranked_b = rank_heads_by_score(scores_a), rank_heads_by_score(scores_b)

rho_split = stats.spearmanr(scores_a.reshape(-1), scores_b.reshape(-1))
print(f"Split-half object generalization ({len(half_a)} vs {len(half_b)} samples, disjoint-by-construction objects):")
print(f"  Spearman rho: {rho_split.correlation:.3f}  (p={rho_split.pvalue:.3e})")
for k in (10, 50, 100):
    overlap = len(top_k_head_set(ranked_a, k) & top_k_head_set(ranked_b, k))
    print(f"  top-{k:<4d} overlap: {overlap}/{k}  ({100 * overlap / k:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(scores_a.reshape(-1), scores_b.reshape(-1), s=8, alpha=0.4, color="tab:purple")
lim = max(float(scores_a.max()), float(scores_b.max())) * 1.05
ax.plot([0, lim], [0, lim], color="tab:red", linestyle="--", linewidth=1, label="y = x")
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel("Vir score, object half A"); ax.set_ylabel("Vir score, object half B")
ax.set_title("Same heads score high on disjoint object sets?"); ax.legend()
save_figure_pdf(fig, outputs.figures_dir / "imagenet_object_generalization.pdf")
plt.show()
print("\nHigh rho / high overlap -> vis heads generalize across arbitrary objects.")
print("Low rho / low overlap  -> vis heads are tied to which specific objects were sampled.")


Split-half object generalization (75 vs 75 samples, disjoint-by-construction objects):
  Spearman rho: 0.998  (p=0.000e+00)
  top-10   overlap: 9/10  (90.0%)
  top-50   overlap: 48/50  (96.0%)
  top-100  overlap: 96/100  (96.0%)

High rho / high overlap -> vis heads generalize across arbitrary objects.
Low rho / low overlap  -> vis heads are tied to which specific objects were sampled.


## Part 6 — Prompt / verb generalization

Same `N_VERB_GRIDS` grids and target cells, evaluated under every phrasing in
`PROMPT_TEMPLATES` (`Find/Locate/Where is/Point to/Identify the {name}.`) plus a
position-only phrasing that never names the object (`ordinal_prompt`, the ImageNet-grid
analogue of comics' panel-pointing query). If the same heads top the ranking under
every *named-object* phrasing, vis heads don't care about the verb — they're
triggered by "there is a reference to a specific object/location to resolve",
independent of wording. Comparing against the position-only phrasing checks whether
that circuitry is object-reference-specific or also fires for pure positional
queries (connecting back to comics, which only ever used positional queries).

In [12]:
verb_rng = np.random.RandomState(SEED + 2)
fixed_grids = []
for _ in range(N_VERB_GRIDS):
    grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=verb_rng,
                        class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
    target_cell = int(verb_rng.randint(N_CELLS))
    fixed_grids.append((grid, target_cell))

verb_scores = {}
for template in PROMPT_TEMPLATES:
    raw, norm, _, valid = discover_vis_head_imagenet(
        model, processor, N_VERB_GRIDS, ROWS, COLS, CELL_SIZE, imagenet_class_dirs, imagenet_class_names,
        SEED, template=template, fixed_grids=fixed_grids,
    )
    verb_scores[template] = raw
    print(f"  [{template:>9s}] valid={valid}/{N_VERB_GRIDS}  mean score={raw.mean():.5f}")

# Position-only phrasing: same grids/targets, but the prompt never names the object.
ordinal_raw_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
ordinal_valid = 0
for grid, target_cell in tqdm(fixed_grids, desc="Vir discovery [imagenet/ordinal]", leave=False):
    prompt = ordinal_prompt(target_cell + 1, N_CELLS)
    try:
        inputs = prepare_inputs(processor, grid.grid, prompt, DEVICE)
        region_ids, _ = assign_grid_cells_to_tokens(
            image_grid_thw=inputs["image_grid_thw"], rows=ROWS, cols=COLS, spatial_merge=spatial_merge)
        attn_at_query = collect_last_query_attentions(model, inputs)
        region_attention = aggregate_region_attention(
            attn_at_query=attn_at_query, inputs=inputs, processor=processor, region_ids=region_ids, n_regions=N_CELLS)
        ordinal_raw_sum += region_attention[:, :, target_cell]
        ordinal_valid += 1
    except Exception as exc:
        print(f"Skipping grid: {exc}")
verb_scores["ordinal (no object name)"] = (ordinal_raw_sum / max(ordinal_valid, 1)).astype(np.float32)
print(f"  [{'ordinal':>9s}] valid={ordinal_valid}/{N_VERB_GRIDS}  mean score={verb_scores['ordinal (no object name)'].mean():.5f}")


Vir discovery [imagenet/find]:   0%|          | 0/30 [00:00<?, ?it/s]

  [     find] valid=30/30  mean score=0.03194


Vir discovery [imagenet/locate]:   0%|          | 0/30 [00:00<?, ?it/s]

  [   locate] valid=30/30  mean score=0.03124


Vir discovery [imagenet/where_is]:   0%|          | 0/30 [00:00<?, ?it/s]

...

  [ point_to] valid=30/30  mean score=0.02651


Vir discovery [imagenet/identify]:   0%|          | 0/30 [00:00<?, ?it/s]

  [ identify] valid=30/30  mean score=0.03094


Vir discovery [imagenet/ordinal]:   0%|          | 0/30 [00:00<?, ?it/s]

  [  ordinal] valid=30/30  mean score=0.03425


In [13]:
template_names = list(verb_scores.keys())
n_t = len(template_names)
overlap_matrix = np.zeros((n_t, n_t))
rho_matrix = np.zeros((n_t, n_t))
K = 50
for i, ti in enumerate(template_names):
    ranked_i = rank_heads_by_score(verb_scores[ti])
    for j, tj in enumerate(template_names):
        ranked_j = rank_heads_by_score(verb_scores[tj])
        overlap_matrix[i, j] = len(top_k_head_set(ranked_i, K) & top_k_head_set(ranked_j, K)) / K
        rho_matrix[i, j] = stats.spearmanr(verb_scores[ti].reshape(-1), verb_scores[tj].reshape(-1)).correlation

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, mat, title in ((axes[0], overlap_matrix, f"Top-{K} head overlap"), (axes[1], rho_matrix, "Spearman rho")):
    im = ax.imshow(mat, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(n_t)); ax.set_xticklabels(template_names, rotation=45, ha="right")
    ax.set_yticks(range(n_t)); ax.set_yticklabels(template_names)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
save_figure_pdf(fig, outputs.figures_dir / "imagenet_verb_generalization.pdf")
plt.show()

named_object_templates = [t for t in template_names if t != "ordinal (no object name)"]
idx = [template_names.index(t) for t in named_object_templates]
named_overlap = overlap_matrix[np.ix_(idx, idx)]
off_diag = named_overlap[~np.eye(len(idx), dtype=bool)]
ordinal_idx = template_names.index("ordinal (no object name)")
named_vs_ordinal = overlap_matrix[idx, ordinal_idx]
print(f"Named-object phrasings, mean pairwise top-{K} overlap (excluding self): {off_diag.mean():.3f}")
print(f"Named-object phrasings vs. position-only phrasing, mean top-{K} overlap: {named_vs_ordinal.mean():.3f}")
print("\nHigh named-object overlap + lower overlap-with-ordinal -> heads are triggered by")
print("'resolve a named-object reference' specifically, independent of the verb used to ask for it.")
print("Named-object overlap ~= ordinal overlap -> the same heads handle any localization query,")
print("named or positional alike (fully general 'resolve a reference to a region' circuitry).")


Named-object phrasings, mean pairwise top-50 overlap (excluding self): 0.804
Named-object phrasings vs. position-only phrasing, mean top-50 overlap: 0.524

High named-object overlap + lower overlap-with-ordinal -> heads are triggered by
'resolve a named-object reference' specifically, independent of the verb used to ask for it.
Named-object overlap ~= ordinal overlap -> the same heads handle any localization query,
named or positional alike (fully general 'resolve a reference to a region' circuitry).


## Verdict

- **Which dataset gives cleaner/more causal vis heads?** Part 3's normalized-score
  comparison says which dataset shows the sharpest, most concentrated attention;
  Part 4's causal-effect sweep says which dataset's actual output depends most on
  those heads — read both together, since (per the base-vs-instruct notebook) they
  can disagree.
- **Are vis heads object-general?** Part 4's split-half test: high Spearman rho and
  top-k overlap between two near-disjoint object sets means yes — the same heads
  handle "find the X" regardless of what X actually is.
- **Are vis heads action/prompt-dependent, or only object-dependent?** Part 5's
  cross-template overlap: high agreement across `Find/Locate/Where is/Point
  to/Identify` means the heads don't care about the verb. Comparing against the
  position-only phrasing (no object name at all) tells you whether the same circuitry
  also handles purely positional queries (connecting back to comics) or is specific to
  resolving a *named* reference.

Rankings for ImageNet-grid are saved to
`logs/vis_head_discovery_compare_datasets_imagenet/`, in the same format as the comics and
COCO rankings, so they drop directly into `interactive_steering_qwen2vl.ipynb`
(`VIS_HEAD_RANKING_PATH`).